# Capítulo 5 · Regresión logística

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_05/04_regresion_logistica.ipynb)

**Aprendizaje supervisado · Parte 4: una probabilidad entre 0 y 1**

### 📚 Índice
1. Cuando la respuesta es sí o no
2. La sigmoide
3. El modelo
4. El costo y el gradiente
5. Un paso a mano
6. Pseudocódigo
7. Una característica, con datos ficticios
8. Dos características
9. Lo mismo con scikit-learn
10. Chuleta resumen


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LogisticRegression

sns.set_theme(style="whitegrid")
palette = sns.color_palette("deep", 4)

SEED = 42
np.random.seed(SEED)


---
## 1. Cuando la respuesta es sí o no

La recta del cuaderno anterior predice un número: puede salir $1.4$ o $-0.2$. Si la respuesta es *aprobó* o *no aprobó*, ese número no es una probabilidad.

La regresión logística usa la misma recta por dentro y la aplasta al intervalo $(0, 1)$. El resultado se lee como $P(y = 1 \mid x)$.


---
## 2. La sigmoide

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Vale $0.5$ cuando $z = 0$, se acerca a $0$ si $z$ es muy negativo y a $1$ si $z$ es muy positivo. Nunca sale del intervalo.


In [ ]:
z = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 4))
sns.lineplot(x=z, y=1 / (1 + np.exp(-z)), color=palette[0])
plt.axhline(0.5, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.xlabel("z")
plt.ylabel("σ(z)")
plt.title("Función sigmoide")
plt.show()


---
## 3. El modelo

$z$ es la recta de siempre. La sigmoide la convierte en probabilidad:

$$z = b + w\, x \qquad \hat{p} = \sigma(z)$$

La clase predicha es $1$ si $\hat{p} \ge 0.5$, y $0$ en caso contrario. Eso ocurre justo cuando $z \ge 0$.

Con dos características, $z = b + w_1 x_1 + w_2 x_2$. La frontera es la recta donde $z = 0$.


---
## 4. El costo y el gradiente

El error cuadrático, compuesto con la sigmoide, deja de ser una cuesta lisa. Se usa la **log loss**. Con $\hat{p}$ recortada para no tomar el logaritmo de cero:

$$
J = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i \log \hat{p}_i + (1 - y_i)\log(1 - \hat{p}_i)\Big]
$$

El gradiente queda en función del error de probabilidad $\hat{p} - y$. No lleva el $2$ del cuadrado:

$$
\frac{\partial J}{\partial b} = \frac{1}{n}\sum (\hat{p}_i - y_i)
\qquad
\frac{\partial J}{\partial w} = \frac{1}{n}\sum (\hat{p}_i - y_i)\, x_i
$$

El paso es el de siempre: restar `learning_rate` por el gradiente. Como $\hat{p}$ está entre $0$ y $1$, un tasa grande no dispara el costo como en el error cuadrático; empuja la sigmoide hacia un escalón.


---
## 5. Un paso a mano

Cuatro puntos: $x = -1, 0, 1, 2$ con clases $y = 0, 0, 1, 1$. Empezamos en $b = 0$, $w = 0$, así que $z = 0$ y $\hat{p} = 0.5$ en los cuatro.

$$
\frac{\partial J}{\partial b} = \frac{0.5+0.5-0.5-0.5}{4} = 0
$$

$$
\frac{\partial J}{\partial w} = \frac{0.5(-1) + 0.5(0) + (-0.5)(1) + (-0.5)(2)}{4} = -0.5
$$

Con $\alpha = 1$:

$$b \leftarrow 0 - 1 \cdot 0 = 0 \qquad w \leftarrow 0 - 1 \cdot (-0.5) = 0.5$$

La ordenada no se mueve: hay tantos ceros como unos y todos se predicen a $0.5$. El peso sí sube, porque los $x$ grandes son de la clase $1$.


In [ ]:
x = np.array([-1.0, 0.0, 1.0, 2.0])
y = np.array([0.0, 0.0, 1.0, 1.0])
b = 0.0
w = 0.0
learning_rate = 1.0

p = np.full(4, 0.5)
grad_b = np.mean(p - y)
grad_w = np.mean((p - y) * x)
b = b - learning_rate * grad_b
w = w - learning_rate * grad_w

print("grad_b:", grad_b, " grad_w:", grad_w)
print("b nuevo:", b, " w nuevo:", w)


---
## 6. Pseudocódigo

```text
learning_rate ← 0.8
b ← 0
w ← 0

repetir pasos veces:
    p ← sigmoide(b + w · x)
    grad_b ← media(p − y)
    grad_w ← media((p − y) · x)
    b ← b − learning_rate · grad_b
    w ← w − learning_rate · grad_w
```

La clase de un punto nuevo es $1$ si `p >= 0.5`.


---
## 7. Una característica, con datos ficticios

Cuarenta alumnos que no aprueban y cuarenta que sí. `horas` está centrada: $0$ es la media del grupo, no «cero horas de estudio». Así la tasa de aprendizaje no depende de la unidad.

Cambia `learning_rate` y vuelve a ejecutar desde la celda del bucle.


In [ ]:
rng = np.random.default_rng(SEED)
horas = np.concatenate([
    rng.normal(-1.2, 1.1, 40),
    rng.normal(1.2, 1.1, 40),
])
aprobo = np.concatenate([np.zeros(40), np.ones(40)])

sns.scatterplot(x=horas, y=aprobo, hue=aprobo, palette=palette[:2], legend=False)
plt.xlabel("horas centradas")
plt.ylabel("aprobó")
plt.show()


In [ ]:
learning_rate = 0.8
pasos = 40

b = 0.0
w = 0.0
costos = []

for _ in range(pasos):
    z = b + w * horas
    p = 1 / (1 + np.exp(-np.clip(z, -20, 20)))
    p = np.clip(p, 1e-8, 1 - 1e-8)
    costos.append(-np.mean(aprobo * np.log(p) + (1 - aprobo) * np.log(1 - p)))
    grad_b = np.mean(p - aprobo)
    grad_w = np.mean((p - aprobo) * horas)
    b = b - learning_rate * grad_b
    w = w - learning_rate * grad_w

clase = (1 / (1 + np.exp(-(b + w * horas))) >= 0.5).astype(int)
aciertos = np.mean(clase == aprobo)
print(f"b = {b:.3f}   w = {w:.3f}   costo = {costos[-1]:.3f}   aciertos = {aciertos:.2f}")


In [ ]:
xs = np.linspace(horas.min(), horas.max(), 200)
prob = 1 / (1 + np.exp(-(b + w * xs)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(x=horas, y=aprobo, hue=aprobo, palette=palette[:2], legend=False, ax=axes[0])
sns.lineplot(x=xs, y=prob, color=palette[2], ax=axes[0])
axes[0].axhline(0.5, color="gray", linestyle="--")
axes[0].set_xlabel("horas centradas")
axes[0].set_ylabel("probabilidad de aprobar")
sns.lineplot(x=np.arange(1, pasos + 1), y=costos, marker="o", color=palette[0], ax=axes[1])
axes[1].set_xlabel("paso")
axes[1].set_ylabel("costo")
plt.tight_layout()
plt.show()


> 🧪 **Prueba tú:** pon `learning_rate = 0.1` y después `4`. La curva de la izquierda se vuelve un escalón cuando la tasa es grande, pero el costo no se dispara: la sigmoide no deja salir de $(0, 1)$.


---
## 8. Dos características

Además de las horas, un segundo dato ficticio: horas de sueño, también centradas. El plano $z = 0$ se ve como una recta en el gráfico.

$$x_2 = -\frac{b + w_1 x_1}{w_2}$$


In [ ]:
rng = np.random.default_rng(SEED)
n = 40
horas = np.concatenate([rng.normal(-1.0, 0.55, n), rng.normal(1.0, 0.55, n)])
sueno = np.concatenate([rng.normal(-0.6, 0.55, n), rng.normal(0.8, 0.55, n)])
aprobo = np.concatenate([np.zeros(n), np.ones(n)])

learning_rate = 0.5
pasos = 80

b = 0.0
w1 = 0.0
w2 = 0.0
costos = []

for _ in range(pasos):
    z = b + w1 * horas + w2 * sueno
    p = 1 / (1 + np.exp(-np.clip(z, -20, 20)))
    p = np.clip(p, 1e-8, 1 - 1e-8)
    costos.append(-np.mean(aprobo * np.log(p) + (1 - aprobo) * np.log(1 - p)))
    grad_b = np.mean(p - aprobo)
    grad_w1 = np.mean((p - aprobo) * horas)
    grad_w2 = np.mean((p - aprobo) * sueno)
    b = b - learning_rate * grad_b
    w1 = w1 - learning_rate * grad_w1
    w2 = w2 - learning_rate * grad_w2

print(f"b = {b:.3f}   w1 = {w1:.3f}   w2 = {w2:.3f}   costo = {costos[-1]:.3f}")

x_linea = np.linspace(horas.min(), horas.max(), 50)
y_linea = -(b + w1 * x_linea) / w2

sns.scatterplot(x=horas, y=sueno, hue=aprobo, palette=palette[:2])
sns.lineplot(x=x_linea, y=y_linea, color=palette[2], label="frontera")
plt.xlabel("horas centradas")
plt.ylabel("sueño centrado")
plt.show()


---
## 9. Lo mismo con scikit-learn

`LogisticRegression` minimiza el mismo costo. `C=np.inf` apaga la regularización, para que no encoja los coeficientes hacia cero.

El bucle de esta celda da 200 pasos, más que los 40 de arriba, para acercarse al mínimo. Los datos son otra vez los de una sola característica: la sección anterior pisó `horas` y `aprobo`.


In [ ]:
rng = np.random.default_rng(SEED)
horas = np.concatenate([
    rng.normal(-1.2, 1.1, 40),
    rng.normal(1.2, 1.1, 40),
])
aprobo = np.concatenate([np.zeros(40), np.ones(40)])

b = 0.0
w = 0.0
for _ in range(200):
    z = b + w * horas
    p = 1 / (1 + np.exp(-np.clip(z, -20, 20)))
    p = np.clip(p, 1e-8, 1 - 1e-8)
    b = b - 0.8 * np.mean(p - aprobo)
    w = w - 0.8 * np.mean((p - aprobo) * horas)

modelo = LogisticRegression(C=np.inf, solver="lbfgs")
modelo.fit(horas.reshape(-1, 1), aprobo)

print(f"bucle     b = {b:.3f}   w = {w:.3f}")
print(f"sklearn   b = {modelo.intercept_[0]:.3f}   w = {modelo.coef_[0, 0]:.3f}")


---
## 10. 📋 Chuleta resumen

| Idea | Código |
|---|---|
| Sigmoide | `1 / (1 + np.exp(-z))` |
| Probabilidad, una característica | `z = b + w * x` y luego la sigmoide |
| Clase predicha | `p >= 0.5` |
| Costo (log loss) | `-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))` |
| Gradiente de $b$ | `np.mean(p - y)` |
| Gradiente de $w$ | `np.mean((p - y) * x)` |
| Un paso | `w = w - learning_rate * grad_w` |
| En scikit-learn | `LogisticRegression(C=np.inf)` |
